# Moduł 5: Dependency Injection w FastAPI

**Ćwiczenie:** #5 - Dependency Injection

---

## Spis treści

1. [Wprowadzenie](#1-wprowadzenie)
2. [Wzorzec Dependency Injection](#2-wzorzec-dependency-injection)
3. [Depends() w FastAPI](#3-depends-w-fastapi)
4. [Common Dependencies](#4-common-dependencies)
5. [Validation Dependencies](#5-validation-dependencies)
6. [Nested Dependencies](#6-nested-dependencies)
7. [Best Practices](#7-best-practices)
8. [Demo Live Coding](#8-demo-live-coding)
9. [Przygotowanie do ćwiczenia](#9-przygotowanie-do-ćwiczenia)
10. [Podsumowanie](#10-podsumowanie)

---

## 1. Wprowadzenie

### Po co Dependency Injection?

Dependency Injection (DI) to wzorzec projektowy pozwalający na:
- **Reusability** - Wielokrotne użycie tej samej logiki
- **DRY (Don't Repeat Yourself)** - Eliminacja duplikacji kodu
- **Testability** - Łatwe mockowanie dependencies w testach
- **Separation of Concerns** - Oddzielenie logiki biznesowej od infrastruktury

### Gdzie to użyjemy w praktyce?

- **Database sessions** - Współdzielenie połączenia do bazy danych
- **Authentication** - Weryfikacja użytkownika w wielu endpointach
- **Pagination** - Wspólna logika stronicowania
- **Validation** - Sprawdzanie czy zasób istnieje (get_or_404)
- **Configuration** - Dostęp do ustawień aplikacji

**Przykład problemu bez DI:**
```python
# Duplikacja kodu w każdym endpoincie
@app.get("/users")
def get_users(skip: int = 0, limit: int = 10):  # ❌
    return users[skip:skip+limit]

@app.get("/posts")
def get_posts(skip: int = 0, limit: int = 10):  # ❌
    return posts[skip:skip+limit]

@app.get("/comments")
def get_comments(skip: int = 0, limit: int = 10):  # ❌
    return comments[skip:skip+limit]
```

**Rozwiązanie z DI:**
```python
# Dependency reusable we wszystkich endpointach
def pagination(skip: int = 0, limit: int = 10):
    return {"skip": skip, "limit": limit}

@app.get("/users")
def get_users(p: dict = Depends(pagination)):  # ✅
    return users[p["skip"]:p["skip"]+p["limit"]]
```

---

## 2. Wzorzec Dependency Injection

### Czym jest Dependency Injection?

**Definicja:** Wzorzec projektowy, w którym zależności (dependencies) są **dostarczane** do obiektu z zewnątrz, zamiast być tworzone wewnątrz.

**Analogia:** Zamiast gotować w restauracji wszystko od zera, dostajesz gotowe składniki od dostawców.

### Przykład bez DI (tightly coupled):
```python
class UserService:
    def __init__(self):
        # UserService tworzy własną zależność - trudno testować!
        self.db = Database("postgresql://...")

    def get_user(self, user_id):
        return self.db.query(user_id)
```

### Przykład z DI (loosely coupled):
```python
class UserService:
    def __init__(self, db: Database):
        # Database dostarczony z zewnątrz - łatwo mockować w testach!
        self.db = db

    def get_user(self, user_id):
        return self.db.query(user_id)

# W produkcji:
service = UserService(db=real_database)

# W testach:
service = UserService(db=mock_database)
```

### Korzyści DI:
1. **Testability** - Możesz podmienić prawdziwą bazę na mock
2. **Flexibility** - Łatwo zmienić implementację bez zmian w kodzie
3. **Reusability** - Ta sama dependency w wielu miejscach
4. **Maintainability** - Łatwiejsze zarządzanie kodem

> **⚠️ WAŻNE: FastAPI "Dependency Injection" vs. Klasyczny DI**
>
> Nazwa "Dependency Injection" w FastAPI może być **myląca**. FastAPI używa tej nazwy, ale **to nie do końca** klasyczny wzorzec DI z design patterns:
>
> | Aspekt | Klasyczny DI (design pattern) | FastAPI `Depends()` |
> |--------|-------------------------------|---------------------|
> | **Główny cel** | **Loosely coupling** między obiektami | **DRY** - nie powtarzaj kodu |
> | **O co chodzi** | Obiekt nie tworzy swoich zależności | Funkcja pomocnicza wielokrotnego użytku |
> | **Typowe użycie** | Separacja warstw, testowanie | Walidacja query params, paginacja, auth |
> | **Prawdziwy DI?** | ✅ Tak | ⚠️ Częściowo |
>
> **Przykład różnicy:**
> ```python
> # Klasyczny DI - obiekt dostaje zależności z zewnątrz
> class UserService:
>     def __init__(self, db: Database):  # ← Injection
>         self.db = db  # Nie tworzy Database sam!
>
> # FastAPI "DI" - głównie DRY i code reuse
> def pagination(page: int = 1, size: int = 10):
>     return {"skip": (page-1)*size, "limit": size}
>
> @app.get("/users")
> def get_users(p: dict = Depends(pagination)):  # ← Reusable helper
>     return users[p["skip"]:p["skip"]+p["limit"]]
> ```
>
> FastAPI używa nazwy "Dependency Injection" bo:
> - Coś jest "wstrzykiwane" do funkcji endpointu
> - Brzmi znajomo dla deweloperów z innych frameworków (Spring, .NET)
> - **Ale główna korzyść to DRY**, nie loosely coupling
>
> W dalszej części notatnika będziemy używać terminu "DI" w kontekście FastAPI.

---

## 3. Depends() w FastAPI

### Jak działa Depends()?

FastAPI automatycznie:
1. **Wywołuje** funkcję dependency
2. **Przekazuje** wynik do twojego endpointu
3. **Waliduje** parametry dependency (jeśli są)
4. **Cachuje** wynik (dla tej samej dependency w jednym requestcie)

### Najprostsza dependency:

### Dependency z walidacją Pydantic (DRY):

**Główna korzyść:** Nie powtarzaj tej samej walidacji query params w wielu endpointach.

In [ ]:
from pydantic import BaseModel, Field

class CommonQueryParams(BaseModel):
    """Pydantic schema dla query params (patrz moduł #3)"""
    q: str | None = None
    skip: int = Field(0, ge=0, description="Przeskocz N elementów")
    limit: int = Field(10, ge=1, le=100, description="Max elementów na stronę")

@app.get("/items")
def get_items(params: CommonQueryParams = Depends()):
    """
    FastAPI automatycznie:
    1. Parsuje query params → CommonQueryParams
    2. Waliduje (skip >= 0, limit 1-100)
    3. Zwraca 422 jeśli błędne wartości
    
    Korzyść DI: Ta sama walidacja w wielu endpointach (DRY)
    """
    return {
        "query": params.q,
        "skip": params.skip,
        "limit": params.limit,
        "items": ["item1", "item2", "item3"][params.skip:params.skip+params.limit]
    }

@app.get("/users")
def get_users(params: CommonQueryParams = Depends()):
    """Ta sama dependency - DRY!"""
    return {
        "params": params.model_dump(),
        "users": ["user1", "user2"][params.skip:params.skip+params.limit]
    }

In [ ]:
from fastapi import Query

def common_params(q: str | None = None, skip: int = 0, limit: int = 10):
    """Dependency z walidacją parametrów"""
    return {"q": q, "skip": skip, "limit": limit}

@app.get("/items")
def get_items(params: dict = Depends(common_params)):
    """
    FastAPI waliduje parametry dependency:
    - q: optional string
    - skip: int (default 0)
    - limit: int (default 10)
    """
    return {"params": params}

@app.get("/users")
def get_users(params: dict = Depends(common_params)):
    """Ta sama dependency - DRY!"""
    return {"params": params}

#### ❓ Pytanie: Czym różni się `params: CommonQueryParams = Depends()` od `params: CommonQueryParams`?

**Krótka odpowiedź:** Dla Pydantic models w FastAPI często **nie ma różnicy** - oba parsują query params.

**Ale są subtelne różnice:**

| Aspekt | `params: Model` | `params: Model = Depends()` |
|--------|-----------------|----------------------------|
| **Pydantic w GET** | Query params (implicit) | Query params (explicit) ✅ |
| **Pydantic w POST** | **Body (JSON)** ⚠️ | Query params (rzadkie!) |
| **Funkcja** | ❌ Nie działa | ✅ Wymaga Depends() |
| **Czytelność** | Mniej jasne | **Bardziej jasne** ✅ |
| **use_cache** | ❌ Brak kontroli | ✅ Można wyłączyć |

**Przykłady:**

```python
# 1. GET request - oba działają identycznie:
@app.get("/users")
def get_users(params: CommonQueryParams):  # ← Query params (implicit)
    pass

@app.get("/users")  
def get_users(params: CommonQueryParams = Depends()):  # ← Query params (explicit) ✅
    pass

# 2. POST request - UWAGA! Różnica:
@app.post("/users")
def create_user(user: UserCreate):  # ← BODY (JSON)
    pass

@app.post("/users")
def create_user(user: UserCreate = Depends()):  # ← Query params (rzadkie!)
    pass

# 3. Funkcja - wymaga Depends():
def get_current_user():
    return {"id": 1}

@app.get("/me")
def me(user: dict = Depends(get_current_user)):  # ✅ OK
    pass

@app.get("/me")
def me(user: get_current_user):  # ❌ To się nie skompiluje!
    pass

# 4. Kontrola cachowania:
@app.get("/profile")
def profile(user: User = Depends(get_current_user, use_cache=False)):  # ✅
    # get_current_user() wywołane za każdym razem (nie cachowane)
    pass
```

**Best practice:** Używaj **`= Depends()`** - jest bardziej explicitne i jasne w intencjach, szczególnie gdy:
- Chcesz pokazać że to dependency (czytelność)
- Używasz funkcji (wymagane)
- Potrzebujesz kontroli nad cachowaniem
- Chcesz być pewny co do źródła danych (body vs query)

**Request:**
```bash
curl "http://localhost:8000/items?q=test&skip=10&limit=20"
# → {"params": {"q": "test", "skip": 10, "limit": 20}}
```

### Dependency jako klasa:

In [ ]:
class CommonQueryParams:
    """Dependency jako klasa (callable)"""
    def __init__(self, q: str | None = None, skip: int = 0, limit: int = 10):
        self.q = q
        self.skip = skip
        self.limit = limit

@app.get("/posts")
def get_posts(params: CommonQueryParams = Depends()):
    """
    Depends() bez argumentu - FastAPI automatycznie użyje klasy z type hint
    """
    return {
        "query": params.q,
        "skip": params.skip,
        "limit": params.limit
    }

---

## 4. Common Dependencies

### 4.1. Pagination Dependency

**Problem:** Stronicowanie powtarza się w wielu endpointach.

**Rozwiązanie:**

In [ ]:
from fastapi import Query

def pagination(
    page: int = Query(1, ge=1, description="Numer strony (od 1)"),
    page_size: int = Query(10, ge=1, le=100, description="Rozmiar strony (1-100)")
):
    """Dependency obliczające skip/limit dla stronicowania"""
    skip = (page - 1) * page_size
    return {"skip": skip, "limit": page_size, "page": page}

# Mock data
all_items = [f"Item {i}" for i in range(1, 101)]

@app.get("/items")
def get_items(p: dict = Depends(pagination)):
    """Pobierz listę items z paginacją"""
    items = all_items[p["skip"]:p["skip"] + p["limit"]]
    return {
        "page": p["page"],
        "page_size": p["limit"],
        "total": len(all_items),
        "items": items
    }

**Request:**
```bash
curl "http://localhost:8000/items?page=2&page_size=10"
# → {
#     "page": 2,
#     "page_size": 10,
#     "total": 100,
#     "items": ["Item 11", "Item 12", ..., "Item 20"]
#   }
```

### 4.2. Filtering Dependency

**Problem:** Filtrowanie danych powtarza się.

**Rozwiązanie:**

In [ ]:
def item_filter(
    category: str | None = Query(None, description="Kategoria"),
    in_stock: bool | None = Query(None, description="Czy w magazynie"),
    min_price: float | None = Query(None, ge=0, description="Minimalna cena")
):
    """Dependency dla filtrowania items"""
    return {
        "category": category,
        "in_stock": in_stock,
        "min_price": min_price
    }

# Mock data
items_db = [
    {"id": 1, "name": "Laptop", "category": "electronics", "price": 1000, "in_stock": True},
    {"id": 2, "name": "Mouse", "category": "electronics", "price": 20, "in_stock": False},
    {"id": 3, "name": "Desk", "category": "furniture", "price": 300, "in_stock": True},
]

@app.get("/items-filtered")
def get_filtered_items(filters: dict = Depends(item_filter)):
    """Pobierz items z filtrowaniem"""
    result = items_db

    if filters["category"]:
        result = [i for i in result if i["category"] == filters["category"]]

    if filters["in_stock"] is not None:
        result = [i for i in result if i["in_stock"] == filters["in_stock"]]

    if filters["min_price"] is not None:
        result = [i for i in result if i["price"] >= filters["min_price"]]

    return {"filters": filters, "count": len(result), "items": result}

**Request:**
```bash
curl "http://localhost:8000/items-filtered?category=electronics&in_stock=true"
# → {"filters": {...}, "count": 1, "items": [{"id": 1, "name": "Laptop", ...}]}
```

### 4.3. Sorting Dependency

In [ ]:
from enum import Enum

class SortOrder(str, Enum):
    asc = "asc"
    desc = "desc"

def sorting(
    sort_by: str = Query("id", description="Pole do sortowania"),
    order: SortOrder = Query(SortOrder.asc, description="Kierunek sortowania")
):
    """Dependency dla sortowania"""
    return {"sort_by": sort_by, "order": order}

@app.get("/items-sorted")
def get_sorted_items(sort: dict = Depends(sorting)):
    """Pobierz posortowane items"""
    result = sorted(
        items_db,
        key=lambda x: x[sort["sort_by"]],
        reverse=(sort["order"] == SortOrder.desc)
    )
    return {"sorting": sort, "items": result}

**Request:**
```bash
curl "http://localhost:8000/items-sorted?sort_by=price&order=desc"
# → Posortowane po cenie malejąco
```

---

## 5. Validation Dependencies

### 5.1. Get or 404 Pattern

**Problem:** Sprawdzanie czy zasób istnieje powtarza się w wielu endpointach.

**Rozwiązanie:**

In [ ]:
from fastapi import HTTPException, status, Path

# Mock database
users_db = {
    1: {"id": 1, "name": "John", "email": "john@example.com"},
    2: {"id": 2, "name": "Jane", "email": "jane@example.com"},
}

def get_user_or_404(user_id: int = Path(..., ge=1)):
    """Dependency sprawdzająca czy user istnieje"""
    user = users_db.get(user_id)
    if not user:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail=f"User {user_id} not found"
        )
    return user

@app.get("/users/{user_id}")
def get_user(user: dict = Depends(get_user_or_404)):
    """Pobierz użytkownika - dependency sprawdza czy istnieje"""
    return user

@app.put("/users/{user_id}")
def update_user(user: dict = Depends(get_user_or_404), name: str = None):
    """Zaktualizuj użytkownika - dependency sprawdza czy istnieje"""
    if name:
        user["name"] = name
    return user

@app.delete("/users/{user_id}")
def delete_user(user: dict = Depends(get_user_or_404)):
    """Usuń użytkownika - dependency sprawdza czy istnieje"""
    users_db.pop(user["id"])
    return {"message": f"User {user['id']} deleted"}

**Request:**
```bash
curl http://localhost:8000/users/1
# → {"id": 1, "name": "John", ...}

curl http://localhost:8000/users/999
# → 404 Not Found: "User 999 not found"
```

**Korzyść:** Logika sprawdzania istnienia zasobu jest w jednym miejscu!

### 5.2. Permission Check Dependency

In [ ]:
def check_admin(role: str = Query(..., description="User role")):
    """Dependency sprawdzająca uprawnienia admin"""
    if role != "admin":
        raise HTTPException(
            status_code=status.HTTP_403_FORBIDDEN,
            detail="Admin access required"
        )
    return {"role": role}

@app.delete("/admin/users/{user_id}")
def admin_delete_user(
    user_id: int,
    admin: dict = Depends(check_admin)
):
    """Tylko admin może usuwać użytkowników"""
    return {"message": f"User {user_id} deleted by {admin['role']}"}

**Request:**
```bash
curl "http://localhost:8000/admin/users/1?role=admin"
# → OK

curl "http://localhost:8000/admin/users/1?role=user"
# → 403 Forbidden
```

---

## 6. Nested Dependencies

### Dependency wywołująca inną dependency

**Dependencies mogą wywoływać inne dependencies:**

In [ ]:
def get_token(token: str = Query(..., description="Auth token")):
    """Dependency poziom 1: Pobierz token"""
    return token

def verify_token(token: str = Depends(get_token)):
    """Dependency poziom 2: Weryfikuj token (zależy od get_token)"""
    if token != "valid_token":
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Invalid token"
        )
    return {"token": token, "valid": True}

def get_current_user(token_data: dict = Depends(verify_token)):
    """Dependency poziom 3: Pobierz użytkownika (zależy od verify_token)"""
    # W prawdziwej aplikacji: query do bazy po user_id z tokenu
    return {"id": 1, "username": "john"}

@app.get("/me")
def get_my_profile(current_user: dict = Depends(get_current_user)):
    """
    FastAPI automatycznie wywołuje cały łańcuch:
    1. get_token() → zwraca token
    2. verify_token(token) → weryfikuje token
    3. get_current_user(token_data) → pobiera użytkownika
    4. endpoint dostaje current_user
    """
    return current_user

**Request:**
```bash
curl "http://localhost:8000/me?token=valid_token"
# → {"id": 1, "username": "john"}

curl "http://localhost:8000/me?token=invalid"
# → 401 Unauthorized
```

### Combining Multiple Dependencies

In [ ]:
@app.get("/items-advanced")
def get_items_advanced(
    pagination: dict = Depends(pagination),
    filters: dict = Depends(item_filter),
    sorting: dict = Depends(sorting)
):
    """
    Endpoint używający wielu dependencies jednocześnie
    """
    # Aplikuj filtry
    result = items_db
    if filters["category"]:
        result = [i for i in result if i["category"] == filters["category"]]

    # Sortuj
    result = sorted(
        result,
        key=lambda x: x[sorting["sort_by"]],
        reverse=(sorting["order"] == SortOrder.desc)
    )

    # Paginacja
    result = result[pagination["skip"]:pagination["skip"] + pagination["limit"]]

    return {
        "pagination": pagination,
        "filters": filters,
        "sorting": sorting,
        "items": result
    }

**Request:**
```bash
curl "http://localhost:8000/items-advanced?page=1&page_size=10&category=electronics&sort_by=price&order=asc"
# → Filtrowane, posortowane i paginowane items
```

---

## 7. Best Practices

### ✅ Rób tak:

**1. Używaj dependencies dla powtarzalnej logiki:**
```python
# ✅ Dobre - reusable dependency
def pagination(page: int = 1, size: int = 10):
    return {"skip": (page-1)*size, "limit": size}

@app.get("/users")
def get_users(p: dict = Depends(pagination)):
    return users[p["skip"]:p["skip"]+p["limit"]]

# ❌ Złe - duplikacja w każdym endpoincie
@app.get("/users")
def get_users(page: int = 1, size: int = 10):
    skip = (page-1)*size
    return users[skip:skip+size]
```

**2. Nazywaj dependencies opisowo:**
```python
# ✅ Dobre - jasna nazwa
def get_current_user():
    pass

def validate_admin_permissions():
    pass

# ❌ Złe - niejasne nazwy
def dep1():
    pass

def check():
    pass
```

**3. Validation w dependencies:**
```python
# ✅ Dobre - walidacja w dependency
def get_user_or_404(user_id: int):
    user = db.get(user_id)
    if not user:
        raise HTTPException(404, "User not found")
    return user

@app.get("/users/{user_id}")
def get_user(user: User = Depends(get_user_or_404)):
    return user  # Już zwalidowany!

# ❌ Złe - walidacja w każdym endpoincie
@app.get("/users/{user_id}")
def get_user(user_id: int):
    user = db.get(user_id)
    if not user:
        raise HTTPException(404, "User not found")
    return user
```

**4. Używaj type hints:**
```python
# ✅ Dobre - type hints
def pagination(page: int = 1, size: int = 10) -> dict:
    return {"skip": (page-1)*size, "limit": size}

# ❌ Złe - brak type hints
def pagination(page=1, size=10):
    return {"skip": (page-1)*size, "limit": size}
```

**5. Dokumentuj dependencies:**
```python
# ✅ Dobre - z docstringiem
def pagination(page: int = Query(1, ge=1), size: int = Query(10, ge=1, le=100)):
    """
    Dependency dla stronicowania

    Args:
        page: Numer strony (od 1)
        size: Rozmiar strony (1-100)

    Returns:
        Dict z skip i limit dla query
    """
    return {"skip": (page-1)*size, "limit": size}
```

### ❌ Nie rób tak:

**1. Nie twórz zbyt skomplikowanych dependencies:**
```python
# ❌ Złe - zbyt dużo logiki
def complex_dependency(a, b, c, d, e, f):
    # 50 linii logiki...
    pass

# ✅ Dobre - rozbij na mniejsze
def simple_dep_1(a, b):
    pass

def simple_dep_2(c, d):
    pass
```

**2. Nie używaj dependencies dla jednorazowej logiki:**
```python
# ❌ Złe - dependency używana tylko raz
def get_special_value():
    return 42

@app.get("/special")
def endpoint(val = Depends(get_special_value)):
    return val

# ✅ Dobre - po prostu użyj w endpoincie
@app.get("/special")
def endpoint():
    return 42
```

**3. Nie modyfikuj globalnego stanu w dependencies:**
```python
# ❌ Złe - modyfikacja globalnego stanu
counter = 0

def increment_counter():
    global counter
    counter += 1  # Side effect!
    return counter

# ✅ Dobre - dependency bez side effects
def get_timestamp():
    return datetime.now()  # Pure function
```

In [ ]:
from fastapi import FastAPI, Depends, HTTPException, Query, Path, status
from typing import Optional
from enum import Enum

app = FastAPI(title="Product API with Dependencies")

# Mock database
products_db = [
    {"id": 1, "name": "Laptop", "category": "electronics", "price": 1000, "in_stock": True},
    {"id": 2, "name": "Mouse", "category": "electronics", "price": 20, "in_stock": False},
    {"id": 3, "name": "Desk", "category": "furniture", "price": 300, "in_stock": True},
    {"id": 4, "name": "Chair", "category": "furniture", "price": 150, "in_stock": True},
]

# Dependencies
class SortOrder(str, Enum):
    asc = "asc"
    desc = "desc"

def pagination(page: int = Query(1, ge=1), page_size: int = Query(10, ge=1, le=100)):
    """Dependency: Pagination"""
    skip = (page - 1) * page_size
    return {"skip": skip, "limit": page_size, "page": page}

def product_filter(
    category: Optional[str] = None,
    in_stock: Optional[bool] = None,
    min_price: Optional[float] = Query(None, ge=0)
):
    """Dependency: Product filtering"""
    return {"category": category, "in_stock": in_stock, "min_price": min_price}

def sorting(sort_by: str = "id", order: SortOrder = SortOrder.asc):
    """Dependency: Sorting"""
    return {"sort_by": sort_by, "order": order}

def get_product_or_404(product_id: int = Path(..., ge=1)):
    """Dependency: Get product or raise 404"""
    product = next((p for p in products_db if p["id"] == product_id), None)
    if not product:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail=f"Product {product_id} not found"
        )
    return product

# Endpoints
@app.get("/products")
def get_products(
    p: dict = Depends(pagination),
    f: dict = Depends(product_filter),
    s: dict = Depends(sorting)
):
    """
    Pobierz listę produktów z filtrowaniem, sortowaniem i paginacją

    **Dependencies:**
    - pagination: page, page_size
    - product_filter: category, in_stock, min_price
    - sorting: sort_by, order
    """
    result = products_db

    # Filtry
    if f["category"]:
        result = [item for item in result if item["category"] == f["category"]]
    if f["in_stock"] is not None:
        result = [item for item in result if item["in_stock"] == f["in_stock"]]
    if f["min_price"] is not None:
        result = [item for item in result if item["price"] >= f["min_price"]]

    # Sortowanie
    result = sorted(
        result,
        key=lambda x: x[s["sort_by"]],
        reverse=(s["order"] == SortOrder.desc)
    )

    # Paginacja
    total = len(result)
    result = result[p["skip"]:p["skip"] + p["limit"]]

    return {
        "page": p["page"],
        "page_size": p["limit"],
        "total": total,
        "filters": f,
        "sorting": s,
        "products": result
    }

@app.get("/products/{product_id}")
def get_product(product: dict = Depends(get_product_or_404)):
    """Pobierz pojedynczy produkt"""
    return product

@app.put("/products/{product_id}")
def update_product(
    product: dict = Depends(get_product_or_404),
    name: Optional[str] = None,
    price: Optional[float] = Query(None, ge=0)
):
    """Zaktualizuj produkt"""
    if name:
        product["name"] = name
    if price is not None:
        product["price"] = price
    return product

@app.delete("/products/{product_id}")
def delete_product(product: dict = Depends(get_product_or_404)):
    """Usuń produkt"""
    products_db.remove(product)
    return {"message": f"Product {product['id']} deleted"}

**Test:**
```bash
# Lista wszystkich produktów
curl http://localhost:8000/products

# Z filtrowaniem i paginacją
curl "http://localhost:8000/products?category=electronics&in_stock=true&page=1&page_size=10&sort_by=price&order=asc"

# Pojedynczy produkt
curl http://localhost:8000/products/1

# 404 dla nieistniejącego
curl http://localhost:8000/products/999

# Update
curl -X PUT "http://localhost:8000/products/1?name=Gaming+Laptop&price=1500"

# Delete
curl -X DELETE http://localhost:8000/products/1
```

---

## 9. Przygotowanie do ćwiczenia

### Ćwiczenie #5: Dependency Injection

**Cel:** Stworzyć API z wieloma dependencies

**Wymagania:**
1. **Pagination dependency** - używana w min. 2 endpointach
2. **Filtering dependency** - min. 2 filtry
3. **Get or 404 dependency** - sprawdzanie czy zasób istnieje
4. **Min. 4 endpointy** używające dependencies
5. **Dokumentacja** - opisać dependencies w Swagger

**Przykładowy endpoint:**
```python
@app.get("/items")
def get_items(
    p: dict = Depends(pagination),
    f: dict = Depends(item_filter)
):
    # Implementacja z użyciem dependencies
    pass
```

**Sprawdzenie:**
```bash
# Uruchom testy
pytest tests/test_exercise_05.py -v

# Sprawdź Swagger
open http://localhost:8000/docs
```

---

## 10. Podsumowanie

### Kluczowe zagadnienia:

1. **Dependency Injection** - wzorzec dostarczania zależności z zewnątrz
2. **Depends()** - mechanizm FastAPI do automatycznego wywoływania dependencies
3. **Reusability** - Ta sama dependency w wielu endpointach (DRY)
4. **Common Dependencies** - Pagination, filtering, sorting
5. **Validation Dependencies** - Get or 404, permission checks
6. **Nested Dependencies** - Dependencies wywołujące inne dependencies
7. **Best Practices** - Opisowe nazwy, type hints, dokumentacja

### Korzyści DI:
- ✅ **DRY** - Brak duplikacji kodu
- ✅ **Testability** - Łatwe mockowanie
- ✅ **Maintainability** - Zmiana w jednym miejscu
- ✅ **Readability** - Czytelny kod endpointów